In [ ]:
from __future__ import annotations
from pathlib import Path
import pickle, fsspec

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst



In [ ]:
fs = fsspec.filesystem("s3")
files = fs.ls("visionlab-members/amarvi/manifold-dynamics/timextime/ed_main")

# full s3 URIs
uris = [f"s3://{f}" for f in files]
local_paths = [vst.fetch(uri) for uri in uris]
rows = []
for fpath in local_paths:
    fpath = Path(fpath)

    with open(fpath, "rb") as f:
        ed = pickle.load(f)
    df_roi = ed.copy()
    rows.append(df_roi)

df = pd.concat(rows, ignore_index=True)

def zscore_group(g):
    mu = g.loc[g.condition == "random", "ED"].mean()
    sigma = g.loc[g.condition == "random", "ED"].std()
    g["ED_z"] = (g["ED"] - mu) / sigma
    return g

df = df.groupby(["roi"], group_keys=False).apply(zscore_group)

In [ ]:
df["target"].unique()

In [ ]:
def target_group(target):
    s = str(target)

    # "middle value" after splitting on "."
    parts = s.split(".")
    middle = parts[1] if len(parts) >= 3 else s

    if middle == "Unknown":
        return "Unknown"
    elif "F" in middle:
        return "F"
    elif "B" in middle:
        return "B"
    elif "O" in middle:
        return "O"
    else:
        return "Other"

df["target_group"] = df["target"].apply(target_group)
df = df[df["condition"] != "random"]

df.groupby(["condition", "target_group"])["ED_z"].describe()

In [ ]:
order = ["F", "B", "O", "Unknown", "Other"]
cond_order = ["global", "local"]
customp = ["#ffa4c7", "#5aa4d1"]

fig, ax = plt.subplots(1,1, figsize=(3,2))
sns.barplot(df, 
            x='target_group', 
            y='ED_z', 
            palette=customp, 
            hue='condition', 
            order=order, 
            hue_order=cond_order,
            errorbar="se",
            errwidth=0.5,
            ax=ax)

ax.set_xticklabels(["Face", "Body", "Object", "Unknown", "Other"])

ax.tick_params(axis="x", labelrotation=45)

ax.set_ylabel("ED (z-score)")
ax.set_ylim(ax.get_ylim())
ax.invert_yaxis()
# ax.legend(frameon=False, loc="upper center", ncol=2)
ax.legend()

sns.despine(ax=ax, trim=True, offset=5)
for label in ax.get_xticklabels():
    label.set_fontsize(12)

# individual dots
sns.stripplot(
    data=df,
    x='target_group',
    y='ED_z',
    hue='condition',
    order=order,
    hue_order=cond_order,
    dodge=True,           # separate by condition
    palette=["black"],
    alpha=0.2,
    size=3,
    legend=False,
    ax=ax
)
ax.set_xlabel("")

save_path = Path.home() / "Downloads" / f"ed_main.png"
plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
target_value = "07.MF1.F"  # or whatever target you want

df_target = pd.concat(
    [r[r["roi"] == target_value] for r in rows],
    ignore_index=True
)
order = ["global", "local", "random"]
fig, ax = plt.subplots(1,1, figsize=(2,2))
sns.boxplot(df_target, x="condition", y="ED", hue="condition", palette=["white"], order=order)
sns.stripplot(df_target, x="condition", y="ED", hue="condition", palette=["black"], alpha=0.05)

ax.set_ylabel("Effective Dimensionality (ED)")

ax.set_xticklabels(["Global", "Local", "Random"])
ax.tick_params(axis="x", labelrotation=45)

ax.set_xlabel("")

sns.despine(ax=ax, trim=True, offset=5)
save_path = Path.home() / "Downloads" / f"ed_{target_value}.svg"
# plt.savefig(save_path, transparent=True, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sns.stripplot(
    data=df[df.condition != "random"],
    x="ED_z",
    y="target",   # or target
    hue="condition",
    dodge=True,
)

plt.axvline(0, linestyle="--", color="black")
plt.xlabel("ED (z-score vs random)")
plt.show()